# Rung 2.2b — retrain on the STEM-FIX + triplet-expanded corpus

Same pipeline as `rung2_colab.ipynb`, on the rebuilt `strips_v2_2`:
- **Tuplet stem/placement bug fixed** — high-note triplets now engrave with the "3" **above** (stems down), matching real Turkish scores.
- **Triplet coverage 3.6×** — +40 triplet-rich pieces (190 total): **23,391 strips, 1,487 triplet strips (6.4%)**, and val now has **89** triplet strips in 8 pieces (was 9), so `\tup3` recall is actually measurable.

**Two ways to train (pick one):**
- **Option A — fine-tune from `rung22-best`** (cell 7A): faster (~4000 steps), but anchored to the old checkpoint (which learned below-only tuplets and never saw the new makams).
- **Option B — retrain from base** (cell 7B): the proven 6000-step recipe from the original pretrained weights. **Recommended here** — the corpus changed materially (new makams, corrected orientation), so a clean run avoids stale bias; ~same cost as Rung 2.2.

**Before running:**
1. Locally: `sh scripts/make_colab_zip.sh` → upload as `MyDrive/tnc/tnc_stemfix_colab.zip` (the rebuilt data).
2. For Option A only: `rung22-best` weights must be on Drive at **`MyDrive/tnc/rung22/best`** (they are, from Rung 2.2).
3. Runtime → Change runtime type → a **GPU** (free: T4; Pro: L4 or A100).
4. Run setup cells (1–5), the **shakeout** once, then EITHER cell 7A or 7B.

Checkpoints stream to **`MyDrive/tnc/rung22-stemfix/`** — the original `rung22/` is left untouched so you can compare.

In [ ]:
# Which GPU did we get? (T4 16GB / L4 24GB / A100 40GB)
!nvidia-smi

In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy the corrected data package Drive -> VM disk and unzip (fast local disk for the dataloader).
%%time
!cp /content/drive/MyDrive/tnc/tnc_stemfix_colab.zip /content/
!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/tnc_stemfix_colab.zip
!ls /content/tnc/src/vision/ && head -c 300 /content/tnc/data/synthetic/strips_v2_2/manifest.jsonl

In [ ]:
# Dependencies (torch + torchvision are preinstalled on Colab).
!pip -q install transformers albumentations opencv-python-headless

In [ ]:
# >>> OPTION A ONLY <<<  Copy the rung22-best start weights Drive -> VM disk.
# SKIP THIS CELL if you're doing Option B (retrain from base) — B needs no rung22 checkpoint.
# rsync skips the 1.1 GB trainer_state.pt (only needed to --resume THAT run). ~574 MB safetensors.
%%time
!mkdir -p /content/rung22-best
!rsync -a --exclude trainer_state.pt /content/drive/MyDrive/tnc/rung22/best/ /content/rung22-best/
!ls -la /content/rung22-best

In [ ]:
# SHAKEOUT (~3 min): 100 tiny steps from BASE (works for either option — it's a wiring smoke test).
# Expect the log `vocab: +25 tokens -> 100 ids` (base), augmentation workers, AMP, checkpoint write
# to Drive; loss must fall. (Option-A users testing the exact start point: add `--model /content/rung22-best`,
# which instead logs `+0 tokens`.) Run once per new setup.
%cd /content/tnc
!python src/vision/train.py \
    --out-dir /content/drive/MyDrive/tnc/rung22-stemfix-shakeout \
    --limit-train 512 --limit-val 128 --max-steps 100 --eval-every 50 --save-every 50 \
    --num-workers 2

In [ ]:
# OPTION A — FINE-TUNE from rung22-best on the rebuilt data (faster). Skip this if you run 7B.
#   T4 ~1.5-2 h | L4 ~50-70 min | A100 ~30-45 min  (at --max-steps 4000).
# On L4/A100 add:  --batch-size 16
%cd /content/tnc
!python src/vision/train.py --model /content/rung22-best \
    --out-dir /content/drive/MyDrive/tnc/rung22-stemfix \
    --lr 2e-5 --max-steps 4000 --num-workers 2

In [ ]:
# OPTION B — RETRAIN FROM BASE (recommended). Drops --model so it starts from the ORIGINAL
# pretrained weights (Flova/omr_transformer) — no rung22 needed, no stale below-tuplet bias.
# Proven Rung-2 recipe: 6000 steps, lr 3e-5. T4 ~2.5-4 h | L4 ~1.5-2 h | A100 ~45-75 min.
# On L4/A100 add:  --batch-size 16
%cd /content/tnc
!python src/vision/train.py \
    --out-dir /content/drive/MyDrive/tnc/rung22-stemfix \
    --lr 3e-5 --max-steps 6000 --num-workers 2

In [ ]:
# RESUME after a disconnect: re-run setup cells 1-4 (and cell 5 only if Option A), then this.
# --resume reloads model+optimizer+scheduler from .../rung22-stemfix/last (ignores --model).
# MATCH the option you ran:  Option A -> --lr 2e-5 --max-steps 4000   |   Option B -> --lr 3e-5 --max-steps 6000
%cd /content/tnc
!python src/vision/train.py --out-dir /content/drive/MyDrive/tnc/rung22-stemfix \
    --lr 3e-5 --max-steps 6000 --num-workers 2 --resume

In [ ]:
# HEADLINE METRIC on the fine-tuned checkpoint: per-class AEU accuracy + \tup3/\tie/\grace recall
# (+ SER, exact match). Compare \tup3 recall vs the Rung-2.2 baseline in src/vision/MODEL_EVAL.md.
%cd /content/tnc
!python src/vision/eval_omr.py --checkpoint /content/drive/MyDrive/tnc/rung22-stemfix/best

## After training

- Download `MyDrive/tnc/rung22-stemfix/best/` locally to `data/checkpoints/rung22-stemfix-best/`.
- Re-run the ONNX export chain (same as Rung 2.2, just new paths): `optimum-cli export onnx` → `quantize_onnx.py` → `onnx_parity.py` (fp32+int8) → `make_browser_gate.py` → browser gate.
- The real test: re-upload the neyzen/notaarsivleri strips in `omr-gate.html` — high-note triplets should now decode `\tup3 … \tupend` instead of `16. 32`.
- Judge on `eval_omr.py`'s per-class table (it appends to `rung22-stemfix/best/eval.jsonl`). If `\tup3` recall on the corrected val set holds and real above-tuplets now read, the fix worked. If not: more steps (`--max-steps 6000 --resume`) or a from-base full retrain (drop `--model`, `--max-steps 6000`).